# KEGG enrichment analysis comparing CPIExtract vs DrugBank

This notebook evaluates how the functional interpretation of matched compounds changes when target annotations are taken from CPIExtract rather than DrugBank.

For each compound, KEGG pathway enrichment is computed from the target set associated with each source. The resulting pathway-level signals are then compared as CPIExtract minus DrugBank enrichment differences. The analysis is repeated across three CPIExtract target definitions: all targets, `ave_pchembl ≥ 3`, and `ave_pchembl ≥ 6`.

The pChEMBL-filtered comparisons use a threshold-matched DrugBank target definition: for each threshold, DrugBank targets are restricted to the subset also present in the CPIExtract target set at that same evidence threshold. This keeps the comparison focused on source-specific enrichment behavior under matched target availability, rather than mixing filtered CPIExtract targets with unfiltered DrugBank targets.

The notebook generates the data and vector figures used for Figure 1 panels G and H.


## How to use this notebook

Run the notebook from top to bottom from within this folder. The default configuration uses cached intermediate files so that the analysis can be reproduced without re-querying external services.

Two switches control recomputation:

- `FORCE_REPARSE_DRUGBANK`: rebuilds the parsed DrugBank target table from the XML ZIP.
- `RUN_ENRICHMENT`: recomputes KEGG enrichment through Enrichr/gseapy.

The default values keep both steps cached. This is the preferred mode when regenerating the comparison tables and figure panels from the archived inputs.


In [ ]:
# If needed in a fresh environment:
# !pip install pandas numpy matplotlib gseapy requests


## 0. Imports and configuration

This section defines paths, evidence thresholds, enrichment settings, and reproducibility switches. The default configuration reads all inputs from the local analysis folder and writes derived tables and figures back to `tables/` and `figures/`.


In [ ]:
from pathlib import Path
import json
import platform
import re
import sys
import time
import warnings
import zipfile
import xml.etree.ElementTree as ET
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    import gseapy as gp
    GSEAPY_AVAILABLE = True
except ImportError:
    gp = None
    GSEAPY_AVAILABLE = False

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    requests = None
    REQUESTS_AVAILABLE = False


BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
TABLE_DIR = BASE_DIR / "tables"
FIG_DIR = BASE_DIR / "figures"
CACHE_DIR = BASE_DIR / "cache"

for directory in [TABLE_DIR, FIG_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CPIE_PATH = DATA_DIR / "CPIE_FB_final_v2.csv"
DRUGBANK_ZIP_PATH = DATA_DIR / "drugbank_all_full_database.xml.zip"
DRUGBANK_CACHE_PATH = CACHE_DIR / "drugbank_compounds_targets_atc_inchikey_v2.csv"
KEGG_ANNOTATION_PATH = CACHE_DIR / "kegg_pathway_annotation_official.csv"

MIN_TARGETS_FOR_ENRICHMENT = 5
SIGNIFICANCE_THRESHOLD = 0.05
GENE_SET_LIBRARY = "KEGG_2021_Human"
ORGANISM = "Human"

CPIE_THRESHOLDS = {
    "all": None,
    "pchembl_ge_3": 3,
    "pchembl_ge_6": 6,
}
THRESHOLD_ORDER = ["all", "pchembl_ge_3", "pchembl_ge_6"]
THRESHOLD_LABELS = {
    "all": "CPIExtract all",
    "pchembl_ge_3": "pChEMBL ≥ 3",
    "pchembl_ge_6": "pChEMBL ≥ 6",
}

RUN_ENRICHMENT = False
FORCE_REPARSE_DRUGBANK = False
SLEEP_BETWEEN_REQUESTS = 0.02
CHECKPOINT_EVERY_COMPOUNDS = 10
MAX_COMPOUNDS_PER_SET = None


## 1. Load and normalize compound-target data

CPIExtract associations are loaded from the full target table and restricted to rows with valid HGNC gene symbols. Compound identifiers are standardized using DrugBank InChIKey when available, falling back to the CPIExtract/PubChem InChIKey otherwise.

DrugBank annotations are loaded from the parsed cache, or rebuilt from the XML ZIP if requested. The parser retains DrugBank IDs, compound names, InChIKeys, ATC codes, and human target gene symbols.


In [ ]:
def clean_str(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return x if x and x.lower() != "nan" else np.nan


def split_pipe_list(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return sorted({v.strip().upper() for v in str(x).split("|") if v.strip()})


def load_cpie(path: Path) -> pd.DataFrame:
    cpie = pd.read_csv(path, low_memory=False)
    for col in ["pc_inchikey", "db_inchikey", "pc_firstblock", "hgnc_symbol"]:
        if col in cpie.columns:
            cpie[col] = cpie[col].map(clean_str)
    if "ave_pchembl" not in cpie.columns:
        raise ValueError("Column 'ave_pchembl' not found in CPIExtract file.")
    cpie["ave_pchembl"] = pd.to_numeric(cpie["ave_pchembl"], errors="coerce")
    cpie["compound_inchikey_for_mapping"] = cpie["db_inchikey"].fillna(cpie["pc_inchikey"])
    cpie["compound_firstblock_for_mapping"] = cpie["compound_inchikey_for_mapping"].str.split("-").str[0]
    cpie = cpie[cpie["hgnc_symbol"].notna()].copy()
    cpie["hgnc_symbol"] = cpie["hgnc_symbol"].astype(str).str.strip().str.upper()
    cpie = cpie[cpie["hgnc_symbol"] != ""]
    return cpie


def get_namespace(tag):
    match = re.match(r"\{(.*)\}", tag)
    return match.group(1) if match else ""


def find_text(element, path, ns):
    found = element.find(path, ns)
    return found.text.strip() if found is not None and found.text else np.nan


def extract_inchikey_from_drug(drug, ns):
    for prop in drug.findall("db:calculated-properties/db:property", ns):
        kind = find_text(prop, "db:kind", ns)
        value = find_text(prop, "db:value", ns)
        if isinstance(kind, str) and kind.lower() == "inchikey":
            return value
    return np.nan


def extract_atc_codes_from_drug(drug, ns):
    codes = []
    for atc in drug.findall("db:atc-codes/db:atc-code", ns):
        code = atc.attrib.get("code")
        if code:
            codes.append(code)
    return sorted(set(codes))


def extract_targets_from_drug(drug, ns):
    targets = []
    for target in drug.findall("db:targets/db:target", ns):
        polypeptide = target.find("db:polypeptide", ns)
        if polypeptide is None:
            continue
        organism = find_text(polypeptide, "db:organism", ns)
        if pd.isna(organism):
            organism = polypeptide.attrib.get("organism", np.nan)
        is_human = True
        if isinstance(organism, str):
            organism_l = organism.lower()
            is_human = ("human" in organism_l) or ("homo sapiens" in organism_l)
        if not is_human:
            continue
        gene = polypeptide.attrib.get("gene-name") or find_text(polypeptide, "db:gene-name", ns)
        if isinstance(gene, str) and gene.strip():
            targets.append(gene.strip().upper())
    return sorted(set(targets))


def parse_drugbank_xml_zip(zip_path: Path) -> pd.DataFrame:
    records = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        xml_names = [name for name in zf.namelist() if name.endswith(".xml")]
        if not xml_names:
            raise ValueError("No XML file found inside DrugBank ZIP.")
        with zf.open(xml_names[0]) as xml_file:
            context = ET.iterparse(xml_file, events=("start", "end"))
            _, root = next(context)
            namespace = get_namespace(root.tag)
            ns = {"db": namespace}
            for event, elem in context:
                if event == "end" and elem.tag == f"{{{namespace}}}drug":
                    primary_id = np.nan
                    for dbid in elem.findall("db:drugbank-id", ns):
                        if dbid.attrib.get("primary") == "true":
                            primary_id = dbid.text.strip() if dbid.text else np.nan
                            break
                    if pd.isna(primary_id):
                        first_id = elem.find("db:drugbank-id", ns)
                        primary_id = first_id.text.strip() if first_id is not None and first_id.text else np.nan
                    name = find_text(elem, "db:name", ns)
                    inchikey = extract_inchikey_from_drug(elem, ns)
                    atc_codes = extract_atc_codes_from_drug(elem, ns)
                    targets = extract_targets_from_drug(elem, ns)
                    records.append({
                        "drugbank_id": primary_id,
                        "drugbank_name": name,
                        "drugbank_inchikey": inchikey,
                        "drugbank_firstblock": str(inchikey).split("-")[0] if isinstance(inchikey, str) else np.nan,
                        "atc_codes": "|".join(atc_codes),
                        "n_atc_codes": len(atc_codes),
                        "drugbank_targets": "|".join(targets),
                        "n_drugbank_targets": len(targets),
                    })
                    elem.clear()
                    root.clear()
    return pd.DataFrame(records)


def load_drugbank() -> pd.DataFrame:
    if FORCE_REPARSE_DRUGBANK and DRUGBANK_CACHE_PATH.exists():
        DRUGBANK_CACHE_PATH.unlink()
    if DRUGBANK_CACHE_PATH.exists():
        return pd.read_csv(DRUGBANK_CACHE_PATH, low_memory=False)
    drugbank = parse_drugbank_xml_zip(DRUGBANK_ZIP_PATH)
    drugbank.to_csv(DRUGBANK_CACHE_PATH, index=False)
    return drugbank


cpie = load_cpie(CPIE_PATH)
drugbank = load_drugbank()


## 2. Build threshold-matched compound sets

For each CPIExtract evidence threshold, the notebook builds one target set per compound. Compounds are then matched to DrugBank by full InChIKey, producing paired CPIExtract and DrugBank target profiles.

For the `ave_pchembl ≥ 3` and `ave_pchembl ≥ 6` analyses, DrugBank targets are intersected with the CPIExtract targets retained at the corresponding threshold. This threshold-matched design reduces bias from comparing target sets with different evidence filters.


In [ ]:
def build_drugbank_target_sets(drugbank_df: pd.DataFrame) -> pd.DataFrame:
    targets = drugbank_df.copy()
    targets["target_list"] = targets["drugbank_targets"].map(split_pipe_list)
    targets["n_targets"] = targets["target_list"].map(len)
    return targets[
        targets["drugbank_inchikey"].notna()
        & (targets["n_targets"] >= MIN_TARGETS_FOR_ENRICHMENT)
    ].copy()


def build_cpie_target_sets(cpie_df: pd.DataFrame, threshold=None) -> pd.DataFrame:
    df = cpie_df.copy()
    if threshold is not None:
        df = df[df["ave_pchembl"] >= threshold].copy()
    grouped = (
        df.dropna(subset=["compound_inchikey_for_mapping", "hgnc_symbol"])
        .groupby("compound_inchikey_for_mapping")
        .agg(
            compound_firstblock=("compound_firstblock_for_mapping", "first"),
            target_list=("hgnc_symbol", lambda x: sorted(set(x.dropna().astype(str)))),
            n_targets=("hgnc_symbol", lambda x: len(set(x.dropna().astype(str)))),
            mean_ave_pchembl=("ave_pchembl", "mean"),
            median_ave_pchembl=("ave_pchembl", "median"),
            n_associations=("hgnc_symbol", "size"),
        )
        .reset_index()
        .rename(columns={"compound_inchikey_for_mapping": "cpie_inchikey"})
    )
    return grouped[grouped["n_targets"] >= MIN_TARGETS_FOR_ENRICHMENT].copy()


def match_cpie_to_drugbank(cpie_targets: pd.DataFrame, drugbank_targets: pd.DataFrame) -> pd.DataFrame:
    db_by_inchikey = (
        drugbank_targets
        .dropna(subset=["drugbank_inchikey"])
        .drop_duplicates("drugbank_inchikey")
        .set_index("drugbank_inchikey", drop=False)
        .to_dict(orient="index")
    )
    rows = []
    for _, row in cpie_targets.iterrows():
        db_info = db_by_inchikey.get(row["cpie_inchikey"])
        if db_info is None:
            continue
        rows.append({
            "compound_inchikey": row["cpie_inchikey"],
            "compound_firstblock": row["compound_firstblock"],
            "drugbank_inchikey": db_info.get("drugbank_inchikey"),
            "drugbank_id": db_info.get("drugbank_id"),
            "compound_name": db_info.get("drugbank_name"),
            "match_type": "full_inchikey",
            "cpie_targets": row["target_list"],
            "n_cpie_targets": row["n_targets"],
            "drugbank_targets": db_info.get("target_list", []),
            "n_drugbank_targets": db_info.get("n_targets", 0),
            "mean_ave_pchembl": row.get("mean_ave_pchembl", np.nan),
            "median_ave_pchembl": row.get("median_ave_pchembl", np.nan),
            "n_cpie_associations": row.get("n_associations", np.nan),
            "atc_codes": db_info.get("atc_codes", ""),
        })
    return pd.DataFrame(rows)


def threshold_match_drugbank_targets(matched_df: pd.DataFrame, threshold_label: str) -> pd.DataFrame:
    tmp = matched_df.copy()
    tmp["drugbank_targets_original"] = tmp["drugbank_targets"]
    tmp["n_drugbank_targets_original"] = tmp["n_drugbank_targets"]
    if threshold_label != "all":
        tmp["drugbank_targets"] = tmp.apply(
            lambda row: sorted(set(row["drugbank_targets"]) & set(row["cpie_targets"])),
            axis=1,
        )
        tmp["n_drugbank_targets"] = tmp["drugbank_targets"].map(len)
    return tmp


def write_matched_table(df: pd.DataFrame, threshold_label: str) -> None:
    tmp = df.copy()
    for col in ["cpie_targets", "drugbank_targets", "drugbank_targets_original"]:
        if col in tmp.columns:
            tmp[col] = tmp[col].map(lambda x: "|".join(x) if isinstance(x, list) else x)
    tmp.to_csv(TABLE_DIR / f"matched_compounds_{threshold_label}_threshold_matched_db.csv", index=False)


drugbank_targets = build_drugbank_target_sets(drugbank)
cpie_target_sets = {
    label: build_cpie_target_sets(cpie, threshold=threshold)
    for label, threshold in CPIE_THRESHOLDS.items()
}
matched_sets = {}
for label in THRESHOLD_ORDER:
    matched = match_cpie_to_drugbank(cpie_target_sets[label], drugbank_targets)
    matched = threshold_match_drugbank_targets(matched, label)
    matched_sets[label] = matched
    write_matched_table(matched, label)


## 3. Run or load KEGG enrichment

KEGG enrichment is performed per compound and per source using `KEGG_2021_Human` through Enrichr/gseapy. Each query is the set of target genes associated with one compound under one target-source definition.

The notebook loads the archived enrichment results by default. Recomputing enrichment is available, but it depends on the external Enrichr service and should be used only when the target sets or enrichment settings change.


In [ ]:
def run_enrichr_for_gene_list(gene_list, gene_sets=GENE_SET_LIBRARY, organism=ORGANISM) -> pd.DataFrame:
    gene_list = sorted({g.strip().upper() for g in gene_list if isinstance(g, str) and g.strip()})
    if len(gene_list) < MIN_TARGETS_FOR_ENRICHMENT:
        return pd.DataFrame()
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=gene_sets,
        organism=organism,
        outdir=None,
        cutoff=1.0,
        verbose=False,
    )
    if enr is None or enr.results is None:
        return pd.DataFrame()
    return enr.results.copy()


def standardize_enrichr_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    return df.rename(columns={c: c.strip().replace(" ", "_").replace("-", "_") for c in df.columns})


def prepare_work_df(compounds_df: pd.DataFrame) -> pd.DataFrame:
    work_df = compounds_df.copy()
    if MAX_COMPOUNDS_PER_SET is not None:
        work_df = work_df.head(MAX_COMPOUNDS_PER_SET)
    return work_df.reset_index(drop=True)


def enrich_compound_set(cache_label: str, source: str, compounds_df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    cache_path = TABLE_DIR / f"enrichment_results_{source}_{cache_label}.csv"
    if not RUN_ENRICHMENT:
        if cache_path.exists():
            return pd.read_csv(cache_path)
        return pd.DataFrame()
    if not GSEAPY_AVAILABLE:
        raise ImportError("gseapy is required to run enrichment. Install it with: pip install gseapy")

    work_df = prepare_work_df(compounds_df)
    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        done = set(cached["compound_inchikey"].dropna().astype(str)) if "compound_inchikey" in cached.columns else set()
        results = [cached] if not cached.empty else []
    else:
        done = set()
        results = []

    remaining_df = work_df[~work_df["compound_inchikey"].astype(str).isin(done)].copy()
    for idx, row in remaining_df.reset_index(drop=True).iterrows():
        try:
            res = standardize_enrichr_columns(run_enrichr_for_gene_list(row[target_col]))
            if not res.empty:
                res["source"] = source
                res["compound_name"] = row["compound_name"]
                res["compound_inchikey"] = row["compound_inchikey"]
                res["drugbank_id"] = row.get("drugbank_id", np.nan)
                res["n_cpie_targets"] = row.get("n_cpie_targets", np.nan)
                res["n_drugbank_targets"] = row.get("n_drugbank_targets", np.nan)
                res["mean_ave_pchembl"] = row.get("mean_ave_pchembl", np.nan)
                res["median_ave_pchembl"] = row.get("median_ave_pchembl", np.nan)
                res["n_cpie_associations"] = row.get("n_cpie_associations", np.nan)
                res["cache_label"] = cache_label
                results.append(res)
        except Exception:
            pass
        time.sleep(SLEEP_BETWEEN_REQUESTS)
        if (idx + 1) % CHECKPOINT_EVERY_COMPOUNDS == 0:
            pd.concat(results, ignore_index=True).to_csv(cache_path, index=False)

    out = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
    out.to_csv(cache_path, index=False)
    return out


def run_or_load_all_enrichment() -> pd.DataFrame:
    all_enrichment_path = TABLE_DIR / "all_enrichment_results_all_thresholds_threshold_matched_db.csv"
    if not RUN_ENRICHMENT and all_enrichment_path.exists():
        return pd.read_csv(all_enrichment_path)

    enrichment_by_threshold = {}
    for label, matched_df in matched_sets.items():
        if matched_df.empty:
            enrichment_by_threshold[label] = pd.DataFrame()
            continue
        cpie_enrichment = enrich_compound_set(label, "CPIExtract", matched_df, "cpie_targets")
        if not cpie_enrichment.empty:
            cpie_enrichment = cpie_enrichment.copy()
            cpie_enrichment["threshold"] = label
            cpie_enrichment["db_target_definition"] = "drugbank_unfiltered_for_all" if label == "all" else "drugbank_intersected_with_cpie_threshold_targets"
        db_cache_label = "drugbank_reference" if label == "all" else f"{label}_db_targets_filtered_by_cpie_threshold"
        db_enrichment = enrich_compound_set(db_cache_label, "DrugBank", matched_df, "drugbank_targets")
        if not db_enrichment.empty:
            db_enrichment = db_enrichment.copy()
            db_enrichment["threshold"] = label
            db_enrichment["db_target_definition"] = "drugbank_unfiltered_for_all" if label == "all" else "drugbank_intersected_with_cpie_threshold_targets"
        enrichment_by_threshold[label] = pd.concat(
            [df for df in [cpie_enrichment, db_enrichment] if not df.empty],
            ignore_index=True,
        ) if (not cpie_enrichment.empty or not db_enrichment.empty) else pd.DataFrame()
        enrichment_by_threshold[label].to_csv(TABLE_DIR / f"enrichment_results_combined_{label}_threshold_matched_db.csv", index=False)

    all_enrichment = pd.concat(
        [df for df in enrichment_by_threshold.values() if not df.empty],
        ignore_index=True,
    ) if any(not df.empty for df in enrichment_by_threshold.values()) else pd.DataFrame()
    if all_enrichment.empty:
        raise ValueError("No enrichment results found. Set RUN_ENRICHMENT=True or check cached enrichment files.")
    all_enrichment.to_csv(all_enrichment_path, index=False)
    return all_enrichment


all_enrichment = run_or_load_all_enrichment()


## 4. Build comparison tables

Adjusted enrichment p-values are transformed to `−log10(adj. p-value)`. To focus on supported pathway signals, non-significant enrichments are set to zero before source comparison.

For each compound, pathway, and threshold, the main statistic is:

`Δ enrichment = −log10(adj. p-value)_CPIExtract − −log10(adj. p-value)_DrugBank`

Positive values indicate stronger pathway enrichment when targets are taken from CPIExtract; negative values indicate stronger enrichment when targets are taken from DrugBank. The same table is also used to classify pathway significance as CPIExtract-only, DrugBank-only, significant in both sources, or neither.


In [ ]:
def adjusted_p_value_column(df: pd.DataFrame) -> str:
    candidates = [c for c in df.columns if c.lower() in ["adjusted_p_value", "adjusted_p-value", "adjusted p-value"]]
    if "Adjusted_P_value" in df.columns:
        return "Adjusted_P_value"
    if "Adjusted_P_Value" in df.columns:
        return "Adjusted_P_Value"
    if candidates:
        return candidates[0]
    raise ValueError("Could not identify adjusted p-value column in enrichment results.")


def build_delta_table(all_enrichment: pd.DataFrame) -> pd.DataFrame:
    adj_col = adjusted_p_value_column(all_enrichment)
    term_col = "Term" if "Term" in all_enrichment.columns else "term"
    df = all_enrichment.copy()
    df[adj_col] = pd.to_numeric(df[adj_col], errors="coerce")
    df["neglog10_adj_p"] = -np.log10(df[adj_col].replace(0, 1e-300))
    df["enrichment_signal"] = np.where(df[adj_col] <= SIGNIFICANCE_THRESHOLD, df["neglog10_adj_p"], 0)
    key_cols = ["threshold", "compound_inchikey", "compound_name", term_col]
    cpie_scores = (
        df[df["source"] == "CPIExtract"][key_cols + ["enrichment_signal"]]
        .drop_duplicates(key_cols)
        .rename(columns={"enrichment_signal": "score_cpie", term_col: "Term"})
    )
    db_scores = (
        df[df["source"] == "DrugBank"][key_cols + ["enrichment_signal"]]
        .drop_duplicates(key_cols)
        .rename(columns={"enrichment_signal": "score_drugbank", term_col: "Term"})
    )
    delta = cpie_scores.merge(db_scores, on=["threshold", "compound_inchikey", "compound_name", "Term"], how="outer")
    delta["score_cpie"] = delta["score_cpie"].fillna(0)
    delta["score_drugbank"] = delta["score_drugbank"].fillna(0)
    delta["delta_cpie_minus_drugbank"] = delta["score_cpie"] - delta["score_drugbank"]
    return delta


def clean_enrichr_kegg_term(term):
    term = str(term)
    term = re.sub(r"\s+Homo sapiens.*$", "", term)
    term = re.sub(r"\s+hsa\d{5}.*$", "", term)
    term = re.sub(r"\s+\(.*?\)$", "", term)
    return term.strip()


def get_kegg_pathway_annotation() -> pd.DataFrame:
    if KEGG_ANNOTATION_PATH.exists():
        return pd.read_csv(KEGG_ANNOTATION_PATH)
    if not REQUESTS_AVAILABLE:
        return pd.DataFrame(columns=["Term_clean", "kegg_category", "kegg_subcategory"])
    response = requests.get("https://rest.kegg.jp/get/br:br08901", timeout=30)
    response.raise_for_status()
    records = []
    current_category = None
    current_subcategory = None
    official_categories = {
        "Metabolism",
        "Genetic Information Processing",
        "Environmental Information Processing",
        "Cellular Processes",
        "Organismal Systems",
        "Human Diseases",
        "Drug Development",
    }
    for line in response.text.splitlines():
        line = line.rstrip()
        if line.startswith("A"):
            candidate = re.sub(r"<.*?>", "", line[1:].strip()).strip()
            if candidate in official_categories:
                current_category = candidate
                current_subcategory = None
        elif line.startswith("B") and current_category is not None:
            current_subcategory = re.sub(r"<.*?>", "", line[1:].strip()).strip() or None
        elif line.startswith("C") and current_category is not None:
            clean = re.sub(r"<.*?>", "", line[1:].strip())
            match = re.match(r"(\d{5})\s+(.+?)(?:\s+\[PATH:map\d{5}\])?$", clean)
            if match:
                records.append({
                    "map_id": "map" + match.group(1),
                    "hsa_id": "hsa" + match.group(1),
                    "Term_base": match.group(2).strip(),
                    "Term_clean": clean_enrichr_kegg_term(match.group(2).strip()),
                    "kegg_category": current_category,
                    "kegg_subcategory": current_subcategory,
                })
    annotation = pd.DataFrame(records).drop_duplicates()
    annotation.to_csv(KEGG_ANNOTATION_PATH, index=False)
    return annotation


def add_kegg_annotation(delta: pd.DataFrame) -> pd.DataFrame:
    annotation = get_kegg_pathway_annotation()
    category_map = annotation.drop_duplicates("Term_clean").set_index("Term_clean")["kegg_category"].to_dict() if not annotation.empty else {}
    subcategory_map = annotation.drop_duplicates("Term_clean").set_index("Term_clean")["kegg_subcategory"].to_dict() if not annotation.empty else {}
    delta = delta.copy()
    delta["Term_clean"] = delta["Term"].map(clean_enrichr_kegg_term)
    delta["kegg_category"] = delta["Term_clean"].map(category_map).fillna("Unmapped")
    delta["kegg_subcategory"] = delta["Term_clean"].map(subcategory_map).fillna("Unmapped")
    return delta


def build_matched_metadata() -> pd.DataFrame:
    rows = []
    for threshold, matched in matched_sets.items():
        if matched.empty:
            continue
        tmp = matched[["compound_inchikey", "compound_name", "atc_codes"]].drop_duplicates().copy()
        tmp["threshold"] = threshold
        rows.append(tmp)
    return pd.concat(rows, ignore_index=True).drop_duplicates() if rows else pd.DataFrame()


def classify_significance(row):
    cpie_sig = row["score_cpie"] > 0
    db_sig = row["score_drugbank"] > 0
    if cpie_sig and db_sig:
        return "Significant in both"
    if cpie_sig and not db_sig:
        return "CPIExtract only"
    if db_sig and not cpie_sig:
        return "DrugBank only"
    return "Neither"


def classify_natural_vs_drug(atc_codes):
    atc = str(atc_codes)
    return "Natural compounds" if ("A11" in atc or "V06" in atc) else "Drug compounds"


delta_long = add_kegg_annotation(build_delta_table(all_enrichment))
matched_metadata = build_matched_metadata()
delta_long = delta_long.merge(matched_metadata, on=["threshold", "compound_inchikey", "compound_name"], how="left")
delta_long["threshold_label"] = delta_long["threshold"].map(THRESHOLD_LABELS).fillna(delta_long["threshold"])
delta_long["significance_class"] = delta_long.apply(classify_significance, axis=1)
delta_long["compound_group"] = delta_long["atc_codes"].fillna("").map(classify_natural_vs_drug)

delta_long.to_csv(TABLE_DIR / "delta_enrichment_cpie_minus_drugbank_long_annotated.csv", index=False)
significance_switching = delta_long.copy()
significance_switching_non_neither = significance_switching[significance_switching["significance_class"] != "Neither"].copy()
significance_switching.to_csv(TABLE_DIR / "significance_switching_cpie_vs_drugbank_by_threshold_FULL.csv", index=False)
significance_switching_non_neither.to_csv(TABLE_DIR / "significance_switching_cpie_vs_drugbank_by_threshold_NON_NEITHER.csv", index=False)


## 5. Figure 1G: Differential KEGG enrichment distribution

Panel G summarizes the distribution of CPIExtract minus DrugBank enrichment differences across all matched compound-pathway pairs. The three curves correspond to the CPIExtract target definitions used in the analysis.

The vertical reference line marks no difference between sources. Values to the right indicate pathways with stronger enrichment under CPIExtract-derived targets.


In [ ]:
def plot_delta_distribution(delta: pd.DataFrame):
    summary = (
        delta.groupby("threshold")["delta_cpie_minus_drugbank"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index()
    )
    summary["percent_positive"] = delta.groupby("threshold")["delta_cpie_minus_drugbank"].apply(lambda x: 100 * (x > 0).mean()).values
    summary["percent_negative"] = delta.groupby("threshold")["delta_cpie_minus_drugbank"].apply(lambda x: 100 * (x < 0).mean()).values
    summary["percent_zero"] = delta.groupby("threshold")["delta_cpie_minus_drugbank"].apply(lambda x: 100 * (x == 0).mean()).values
    summary.to_csv(TABLE_DIR / "figure_1_panel_G_delta_distribution_summary_by_threshold.csv", index=False)

    finite_values = delta["delta_cpie_minus_drugbank"].replace([np.inf, -np.inf], np.nan).dropna()
    if finite_values.empty:
        raise ValueError("No finite delta values are available for plotting.")

    x_min = np.floor(finite_values.min() / 10) * 10
    x_max = np.ceil(finite_values.max() / 10) * 10
    bins = np.linspace(x_min, x_max, 100)

    labels = {
        "all": "All",
        "pchembl_ge_3": "pChEMBL ≥ 3",
        "pchembl_ge_6": "pChEMBL ≥ 6",
    }
    colors = {
        "all": "#1f77b4",
        "pchembl_ge_3": "#ff7f0e",
        "pchembl_ge_6": "#2ca02c",
    }

    fig, ax = plt.subplots(figsize=(8.8, 5.2))

    for threshold in THRESHOLD_ORDER:
        values = (
            delta.loc[delta["threshold"].eq(threshold), "delta_cpie_minus_drugbank"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )
        ax.hist(
            values,
            bins=bins,
            histtype="step",
            linewidth=2,
            color=colors[threshold],
            label=f"{labels[threshold]} (n={len(values):,})",
        )

    ax.axvline(0, linestyle="--", linewidth=0.8, color="black")
    ax.set_yscale("log")
    ax.set_xlim(x_min, x_max)
    ax.set_title("Distribution of differential KEGG enrichment")
    ax.set_xlabel("Δ −log10(adj. p-value)\nCPIExtract − DrugBank")
    ax.set_ylabel("Number of compound–pathway pairs (log scale)")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="Compound-t filtering", frameon=True, loc="upper right")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "figure_1_panel_G_delta_kegg_enrichment_distribution.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / "figure_1_panel_G_delta_kegg_enrichment_distribution.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "figure_1_panel_G_delta_kegg_enrichment_distribution.svg", bbox_inches="tight")
    plt.show()


plot_delta_distribution(delta_long)


## 6. Figure 1H: Significance switching

Panel H summarizes how significant pathway enrichments are assigned across sources. Compound-pathway pairs are grouped as CPIExtract-only, DrugBank-only, or significant in both sources.

The panel separates compounds with nutritional/natural ATC annotations (`A11` or `V06`) from the remaining matched compounds, allowing the source-specific enrichment behavior to be compared between these two compound groups.


In [ ]:
def build_switching_percent_table(significance_df: pd.DataFrame) -> pd.DataFrame:
    class_order = ["CPIExtract only", "DrugBank only", "Significant in both"]
    counts = (
        significance_df
        .groupby(["compound_group", "threshold", "significance_class"])
        .size()
        .reset_index(name="count")
    )
    counts_pivot = (
        counts
        .pivot_table(
            index=["compound_group", "threshold"],
            columns="significance_class",
            values="count",
            fill_value=0,
        )
        .reindex(
            pd.MultiIndex.from_product(
                [["Natural compounds", "Drug compounds"], THRESHOLD_ORDER],
                names=["compound_group", "threshold"],
            ),
            fill_value=0,
        )
        .reindex(columns=class_order, fill_value=0)
    )
    percent = counts_pivot.copy().astype(float)
    total = percent.sum(axis=1)
    percent = percent.div(total.replace(0, np.nan), axis=0).mul(100).fillna(0)
    counts_pivot.reset_index().to_csv(TABLE_DIR / "figure_1_panel_H_significance_switching_counts.csv", index=False)
    percent.reset_index().to_csv(TABLE_DIR / "figure_1_panel_H_significance_switching_percent.csv", index=False)
    return percent


def plot_significance_switching(significance_df: pd.DataFrame):
    class_order = ["CPIExtract only", "DrugBank only", "Significant in both"]
    class_colors = {
        "CPIExtract only": "#B2182B",
        "DrugBank only": "#2166AC",
        "Significant in both": "#4D4D4D",
    }
    percent = build_switching_percent_table(significance_df)

    fig, ax = plt.subplots(figsize=(10.5, 6))
    x = np.arange(len(THRESHOLD_ORDER))
    bar_width = 0.22
    offsets = np.linspace(-bar_width, bar_width, len(class_order))

    for offset, cls in zip(offsets, class_order):
        natural_vals = percent.loc[("Natural compounds", slice(None)), cls].values
        drug_vals = percent.loc[("Drug compounds", slice(None)), cls].values
        ax.bar(x + offset, natural_vals, width=bar_width, color=class_colors[cls], edgecolor="black", linewidth=0.4, label=cls)
        ax.bar(x + offset, -drug_vals, width=bar_width, color=class_colors[cls], edgecolor="black", linewidth=0.4)

    ax.axhline(0, color="black", linewidth=1)
    ax.set_ylim(-105, 105)
    ax.set_yticks([-100, -75, -50, -25, 0, 25, 50, 75, 100])
    ax.set_yticklabels(["100", "75", "50", "25", "0", "25", "50", "75", "100"])
    ax.set_xticks(x)
    ax.set_xticklabels([THRESHOLD_LABELS[t] for t in THRESHOLD_ORDER])
    ax.set_xlabel("CPIExtract target filtering")
    ax.set_ylabel("Percentage of significant compound-pathway pairs (%)")
    ax.text(0.01, 0.96, "Natural compounds", transform=ax.transAxes, fontsize=11, fontweight="bold", ha="left", va="top")
    ax.text(0.01, 0.04, "Drug compounds", transform=ax.transAxes, fontsize=11, fontweight="bold", ha="left", va="bottom")
    ax.set_title("Significance switching between CPIExtract and DrugBank\nNatural compounds versus drug compounds")
    ax.grid(axis="y", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(title="Significance class", frameon=True, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "figure_1_panel_H_significance_switching_natural_vs_drug.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / "figure_1_panel_H_significance_switching_natural_vs_drug.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "figure_1_panel_H_significance_switching_natural_vs_drug.svg", bbox_inches="tight")
    plt.show()


plot_significance_switching(significance_switching_non_neither)


## 7. Reproducibility metadata and run summary

The final cell records the analysis settings, input paths, row counts, package versions, and figure locations. It also prints a compact summary so that a completed run can be checked without inspecting intermediate tables.


In [ ]:
reproducibility_summary = {
    "base_dir": str(BASE_DIR),
    "cipextract_file": str(CPIE_PATH),
    "drugbank_zip_file": str(DRUGBANK_ZIP_PATH),
    "drugbank_cache_file": str(DRUGBANK_CACHE_PATH),
    "enrichment_library": GENE_SET_LIBRARY,
    "minimum_targets_for_enrichment": MIN_TARGETS_FOR_ENRICHMENT,
    "significance_threshold": SIGNIFICANCE_THRESHOLD,
    "run_enrichment": RUN_ENRICHMENT,
    "thresholds": CPIE_THRESHOLDS,
    "n_cpie_rows": int(len(cpie)),
    "n_drugbank_rows": int(len(drugbank)),
    "n_enrichment_rows": int(len(all_enrichment)),
    "n_delta_rows": int(len(delta_long)),
    "n_significant_pairs": int(len(significance_switching_non_neither)),
}
with open(TABLE_DIR / "kegg_enrichment_reproducibility_summary.json", "w") as f:
    json.dump(reproducibility_summary, f, indent=2)

session_info = {"python": sys.version, "platform": platform.platform()}
for package in ["pandas", "numpy", "matplotlib", "gseapy", "requests"]:
    try:
        session_info[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        session_info[package] = "not installed"
with open(TABLE_DIR / "kegg_enrichment_session_info.json", "w") as f:
    json.dump(session_info, f, indent=2)

panel_g_svg = FIG_DIR / "figure_1_panel_G_delta_kegg_enrichment_distribution.svg"
panel_h_svg = FIG_DIR / "figure_1_panel_H_significance_switching_natural_vs_drug.svg"

print("CPIExtract vs DrugBank KEGG enrichment run complete")
print(f"Mode: {'recomputed enrichment' if RUN_ENRICHMENT else 'loaded cached enrichment'}")
print(f"Input rows: CPIExtract={len(cpie):,}; DrugBank parsed compounds={len(drugbank):,}")
print(f"Comparison table: {len(delta_long):,} compound-pathway rows; {len(significance_switching_non_neither):,} significant rows")
print(f"Panel G SVG: {panel_g_svg}")
print(f"Panel H SVG: {panel_h_svg}")
